# CNN layers

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        # Conv layers
        self.conv1 = nn.Conv2d(3, 6, 5)   # 3→6 channels, 5×5 kernel
        self.pool  = nn.MaxPool2d(2, 2)   # 2×2 max pool, stride=2
        self.conv2 = nn.Conv2d(6, 16, 5)  # 6→16 channels, 5×5 kernel
        
        # Fully connected layers
        self.fc1 = nn.Linear(16 * 5 * 5, 120)  # 16*5*5 comes from conv output size
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)  # 10 classes

    def forward(self, x):
        # Input shape: (batch, 3, 32, 32)
        x = self.pool(F.relu(self.conv1(x)))  # → (batch, 6, 14, 14)
        x = self.pool(F.relu(self.conv2(x)))  # → (batch, 16, 5, 5)
        x = torch.flatten(x, 1)               # → (batch, 16*5*5)
        x = F.relu(self.fc1(x))               # → (batch, 120)
        x = F.relu(self.fc2(x))               # → (batch, 84)
        x = self.fc3(x)                       # → (batch, 10)
        return x

net = Net()


# loss fxn and optimizer

In [ ]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()  # Good for multi-class classification
optimizer = optim.SGD(net.parameters(), lr=0.001, momentum=0.9)


# training loop

In [ ]:
for epoch in range(2):  # loop over the dataset 2 times
    running_loss = 0.0
    for i, data in enumerate(trainloader, 0):
        # Get inputs and labels
        inputs, labels = data

        # Zero gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = net(inputs)
        loss = criterion(outputs, labels)

        # Backward pass and update weights
        loss.backward()
        optimizer.step()

        # Print loss every 2000 mini-batches
        running_loss += loss.item()
        if i % 2000 == 1999:
            print(f'[{epoch + 1}, {i + 1:5d}] loss: {running_loss / 2000:.3f}')
            running_loss = 0.0

print('Finished Training')


# saving model

In [ ]:
# Save
PATH = './cifar_net.pth'
torch.save(net.state_dict(), PATH)

# Load (later)
net = Net()
net.load_state_dict(torch.load(PATH, weights_only=True))


# testing and eval

In [ ]:
# Test accuracy
correct = 0
total = 0
with torch.no_grad():
    for data in testloader:
        images, labels = data
        outputs = net(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f'Accuracy of the network on the 10000 test images: {100 * correct / total:.1f} %')

# Per-class accuracy
correct_pred = {classname: 0 for classname in classes}
total_pred = {classname: 0 for classname in classes}

with torch.no_grad():
    for data in testloader:
        images, labels = data
        outputs = net(images)
        _, predictions = torch.max(outputs, 1)
        for label, prediction in zip(labels, predictions):
            if label == prediction:
                correct_pred[classes[label]] += 1
            total_pred[classes[label]] += 1

for classname, correct_count in correct_pred.items():
    accuracy = 100 * float(correct_count) / total_pred[classname]
    print(f'Accuracy for class: {classname:5s} is {accuracy:.1f} %')


# making visuals

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def imshow(img):
    img = img / 2 + 0.5  # unnormalize from [-1,1] to [0,1]
    npimg = img.numpy()
    plt.imshow(np.transpose(npimg, (1, 2, 0)))
    plt.show()

# Get one batch from test set
dataiter = iter(testloader)
images, labels = next(dataiter)

# Make predictions
outputs = net(images)
_, predicted = torch.max(outputs, 1)

# Show images and predictions
imshow(torchvision.utils.make_grid(images))
print('GroundTruth: ', ' '.join(f'{classes[labels[j]]:5s}' for j in range(4)))
print('Predicted:   ', ' '.join(f'{classes[predicted[j]]:5s}' for j in range(4)))
